In [ ]:
import os
import mysql.connector
from dotenv import load_dotenv

# Cargamos las variables del archivo .env
load_dotenv()

if not os.getenv("DB_PORT"):
    print("⚠️ ATENCIÓN: No se pudo leer el archivo .env.")
else:
    try:
        # Intentar conectar a la base de datos
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD")
        )
        
        if conexion.is_connected():
            print("¡Conexión exitosa a la base de datos BiblioIA!")
            
            # Acá está el cambio: usamos server_info sin paréntesis
            info_servidor = conexion.server_info 
            print(f"Versión del servidor MySQL: {info_servidor}")
            
            conexion.close()
            
    except mysql.connector.Error as err:
        print(f"Error al conectar a MySQL: {err}")

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Cargamos el .env
load_dotenv(override=True)

# Configuramos el cliente para Groq
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("LLM_API_KEY")
)

# Prueba de conexión actualizada
print("Probando conexión a Groq...")
try:
    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": "Hola, ¿estás ahí?"}],
        # Usamos un modelo que sí está activo actualmente
        model="llama-3.3-70b-versatile",
    )
    print("🤖 Respuesta de la IA:")
    print(chat_completion.choices[0].message.content)
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Cargamos el .env
load_dotenv(override=True)

# Configuramos el cliente para Groq
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("LLM_API_KEY")
)

def consultar_biblioia(pregunta_usuario):
    # 1. Prompt de Sistema Robusto (Requisito del TP)
    prompt_sistema = """
    Sos un asistente experto en bases de datos MySQL para una biblioteca. 
    Tu objetivo es traducir preguntas en lenguaje natural a consultas SQL precisas.

    ESQUEMA DE LA BASE DE DATOS (Descripción de tablas y columnas):
    - Libro (Isbn, Titulo, Año, StockTotal, StockDisponible)
    - Socio (Id, Dni, Nombre, Apellido, Mail, FechaAlta, IdSexo, IdEstadoSocio)
    - Ejemplar (Numero, IsbnLibro, UbicacionFisica, IdEstadoEjemplar)
    - Prestamo (Id, IdSocio, IdEjemplar, FechaPrestamo, FechaVencimiento, FechaDevolucion, IdEstadoPrestamo)
    - Sancion (Id, FechaInicio, FechaFin, Motivo, IdTipoSancion, IdPrestamo, IdSocio)
    - Autor_Libro (IdAutor, IsbnLibro)
    - GeneroLibro_Libro (IdGeneroLibro, IsbnLibro)

    CONSTRAINTS Y REGLAS DE NEGOCIO (Estados):
    - EstadoSocio: 1='Activo', 2='Suspendido', 3='Baja'
    - EstadoEjemplar: 1='Disponible', 2='Prestado', 3='Baja'
    - EstadoPrestamo: 1='Activo', 2='Devuelto', 3='Vencido'
    - Para saber si un préstamo está vencido, filtrar por IdEstadoPrestamo = 3.

    EJEMPLOS DE PREGUNTAS Y SQL (Few-Shot Prompting):

    Pregunta: "¿Cuáles son los 5 libros más prestados?"
    Respuesta: SELECT l.Titulo, COUNT(p.Id) AS TotalPrestamos FROM Libro l JOIN Ejemplar e ON l.Isbn = e.IsbnLibro JOIN Prestamo p ON e.Numero = p.IdEjemplar GROUP BY l.Isbn, l.Titulo ORDER BY TotalPrestamos DESC LIMIT 5;

    Pregunta: "¿Qué socios tienen préstamos vencidos en este momento?"
    Respuesta: SELECT s.Nombre, s.Apellido, s.Mail FROM Socio s JOIN Prestamo p ON s.Id = p.IdSocio WHERE p.IdEstadoPrestamo = 3;

    Pregunta: "¿Cuántos ejemplares disponibles tenemos en total?"
    Respuesta: SELECT COUNT(*) AS TotalDisponibles FROM Ejemplar WHERE IdEstadoEjemplar = 1;

    INSTRUCCIÓN CRÍTICA:
    Respondé ÚNICA Y EXCLUSIVAMENTE con una consulta SQL válida para MySQL que responda a la pregunta del usuario. NO incluyas explicaciones, saludos, notas, ni bloques de código Markdown (```sql ... ```). Devuelve solo la cadena de texto de la consulta.
    """
    
    # 2. Llamamos a Groq enviando el sistema y luego la pregunta del usuario
    try:
        chat_completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": pregunta_usuario}
            ],
            temperature=0.1 # Temperatura baja para que sea lógico y no creativo
        )
        
        sql_generado = chat_completion.choices[0].message.content.strip()
        
        # Pequeña limpieza por si la IA devuelve saltos de línea extra
        sql_generado = sql_generado.replace("```sql", "").replace("```", "").strip()
        
        return sql_generado
    
    except Exception as e:
        return f"Error al generar SQL: {e}"

# --- PRUEBA DEL AGENTE ---
pregunta_test = "¿Cuántos socios tenemos suspendidos actualmente?"
print(f"Pregunta: {pregunta_test}")
print("Generando SQL...")

sql_resultante = consultar_biblioia(pregunta_test)
print(f"\nConsulta SQL Generada:\n{sql_resultante}")